# BNA-Eval

Единая точка запуска оценки BlocksNetAgent по методике `docs/evaluation/README.md`: вопросы, прогоны агента, авто-метрики, LLM-судья, скоркарта и ручная проверка трейса.

Перед запуском проверьте `.env` с `FP2MP_CHAT_URL`/`FP2MP_API_KEY` или `CHAT_URL`/`API_KEY`.

In [1]:
MODEL = 'openai/gpt-4o'
N_RUNS = 3
M_JUDGES = 5
JUDGE_MODEL = None
QUESTIONS = 'docs/bench/questions.yaml'
CATEGORIES = None  # например ['A', 'B']
REUSE_RESULTS = None  # путь к готовому docs/bench/results_*.csv
REUSE_JUDGE = None  # путь к готовому docs/bench/judge_*.csv
RUN_JUDGE = False
MAX_ITERATIONS = 10

In [2]:
from pathlib import Path
import json
import sys
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'blocksnet_agent').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
load_dotenv(ROOT / '.env')

from scripts.run_bench import DEFAULT_RESULTS_DIR, _load_questions, run_benchmark
from scripts.eval_judge import case_from_result_row, judge_case, judges_to_long_rows, write_csv as write_judge_csv
from scripts.build_scorecard import build_scorecard, build_scorecard_frames

bench_dir = ROOT / 'docs' / 'bench'
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
questions_path = ROOT / QUESTIONS
bench_dir

WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench')

In [3]:
questions = _load_questions(questions_path)
if CATEGORIES:
    allowed = {item.upper() for item in CATEGORIES}
    questions = [q for q in questions if str(q.get('category', '')).upper() in allowed]
questions_df = pd.DataFrame(questions)
questions_df[['id', 'category', 'question', 'expected_tools']]

,id,category,question,expected_tools
0,q01_block_603_development,A,Что нужно разместить в квартале 603?,"[get_block_info, get_weakest_services, propose..."
1,q02_block_7_needs,A,Чего не хватает в квартале 7?,"[get_block_info, get_weakest_services]"
2,q03_block_711_provision,A,Какова обеспеченность квартала 711 сервисами?,"[get_block_info, get_weakest_services]"
3,q04_city_hotel_placement,B,Где разместить новые гостиницы?,"[compute_service_provision, suggest_target_blo..."
4,q05_city_pitch_placement,B,Где разместить спортивные площадки?,"[compute_service_provision, suggest_target_blo..."
5,q06_city_pharmacy_gaps,B,Где не хватает аптек?,"[compute_service_provision, suggest_target_blo..."
6,q07_pedestrian_streets,E/D,Какие улицы сделать пешеходными?,"[compute_connectivity, compute_mean_accessibil..."
7,q08_healthcare_gaps,C,Какие кварталы хуже всего обеспечены объектами...,"[compute_service_provision, suggest_target_blo..."
8,q09_lowest_transport_accessibility,C,Где самая низкая транспортная доступность?,[compute_mean_accessibility]
9,q10_least_connected_blocks,C,Какие кварталы наименее связны?,[compute_connectivity]


In [4]:
if REUSE_RESULTS:
    results_path = Path(REUSE_RESULTS)
    if not results_path.is_absolute():
        results_path = ROOT / results_path
else:
    results_path = bench_dir / f'results_{stamp}.csv'
    run_benchmark(
        questions_path=questions_path,
        n=N_RUNS,
        output_path=results_path,
        model=MODEL,
        max_iterations=MAX_ITERATIONS,
        categories=CATEGORIES,
    )
results_df = pd.read_csv(results_path)
results_path, results_df.head()

2026-06-17 00:44:18.151 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:44:18.153 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-06-17 00:44:18.166 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-06-17 00:44:18.177 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving LP problems until max depth or break condition reached
2026-06-17 00:44:18.369 | SUCCESS  | blocksnet.analysis.provision.competitive.core:competitive_provision:203 - Provision assessment finished
2026-06-17 00:44:18.378 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:44:18.379 | WARNING  | blocksnet.analysi

q01_block_603_development #1: D1=1.00, D2=0.75, D3=0.56, D4=0.72, category=A, selection=0.25, elapsed=48.1s


2026-06-17 00:45:04.866 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:45:04.867 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-06-17 00:45:04.876 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-06-17 00:45:04.886 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving LP problems until max depth or break condition reached
2026-06-17 00:45:05.492 | SUCCESS  | blocksnet.analysis.provision.competitive.core:competitive_provision:203 - Provision assessment finished
2026-06-17 00:45:05.769 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:45:05.771 | WARNING  | blocksnet.analysi

q01_block_603_development #2: D1=1.00, D2=0.67, D3=0.52, D4=0.76, category=A, selection=0.0, elapsed=39.5s


2026-06-17 00:45:43.034 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:45:43.034 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-06-17 00:45:43.046 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-06-17 00:45:43.057 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving LP problems until max depth or break condition reached
2026-06-17 00:45:43.652 | SUCCESS  | blocksnet.analysis.provision.competitive.core:competitive_provision:203 - Provision assessment finished
2026-06-17 00:45:43.912 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 00:45:43.913 | WARNING  | blocksnet.analysi

...

2026-06-17 01:31:10.373 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 01:31:10.374 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-06-17 01:31:10.383 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-06-17 01:31:10.394 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving LP problems until max depth or break condition reached
2026-06-17 01:31:10.569 | SUCCESS  | blocksnet.analysis.provision.competitive.core:competitive_provision:203 - Provision assessment finished
2026-06-17 01:31:10.575 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 01:31:10.576 | WARNING  | blocksnet.analysi

q18_block_243_development #2: D1=1.00, D2=0.58, D3=0.56, D4=0.59, category=A, selection=0.25, elapsed=71.8s


2026-06-17 01:32:23.303 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 01:32:23.304 | WARNING  | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:33 - No demand in columns. Imputing using population column and demand parameter
2026-06-17 01:32:23.314 | INFO     | blocksnet.analysis.provision.competitive.core:_supply_self:56 - Supplying blocks with own capacities
2026-06-17 01:32:23.330 | INFO     | blocksnet.analysis.provision.competitive.core:competitive_provision:190 - Setting and solving LP problems until max depth or break condition reached
2026-06-17 01:32:23.965 | SUCCESS  | blocksnet.analysis.provision.competitive.core:competitive_provision:203 - Provision assessment finished
2026-06-17 01:32:24.235 | INFO     | blocksnet.analysis.provision.competitive.core:_initialize_provision_df:29 - Initializing provision DataFrame
2026-06-17 01:32:24.237 | WARNING  | blocksnet.analysi

q18_block_243_development #3: D1=1.00, D2=0.76, D3=0.76, D4=0.79, category=A, selection=0.35, elapsed=160.6s


(WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench/results_20260617_004403.csv'),
                           id category class  repeat  \
 0  q01_block_603_development        A     A       1   
 1  q01_block_603_development        A     A       2   
 2  q01_block_603_development        A     A       3   
 3          q02_block_7_needs        A     A       1   
 4          q02_block_7_needs        A     A       2   
 
                                question expected_entity  expects_grounding  \
 0  Что нужно разместить в квартале 603?             603               True   
 1  Что нужно разместить в квартале 603?             603               True   
 2  Что нужно разместить в квартале 603?             603               True   
 3         Чего не хватает в квартале 7?               7               True   
 4         Чего не хватает в квартале 7?               7               True   
 
    expects_measured  expects_out_of_model  \
 0              True                 False   
 1

In [5]:
dimension_cols = ['D1', 'D2', 'D3', 'D4', 'composite']
available = [col for col in dimension_cols if col in results_df.columns]
results_df[['id', 'category', 'repeat', *available]].head()

,id,category,repeat,D1,D2,D3,D4,composite
0,q01_block_603_development,A,1,1.0,0.750000,0.555556,0.724,0.757389
1,q01_block_603_development,A,2,1.0,0.666667,0.518519,0.756,0.735296
2,q01_block_603_development,A,3,1.0,0.750000,0.518519,0.756,0.756130
3,q02_block_7_needs,A,1,1.0,0.833333,0.518519,0.710,0.765463
4,q02_block_7_needs,A,2,1.0,0.666667,0.518519,0.756,0.735296


In [6]:
judge_path = None
judge_rows = []
if REUSE_JUDGE:
    judge_path = Path(REUSE_JUDGE)
    if not judge_path.is_absolute():
        judge_path = ROOT / judge_path
elif RUN_JUDGE:
    judge_path = bench_dir / f'judge_{stamp}.csv'
    for row in results_df.to_dict('records'):
        case = case_from_result_row(row)
        evaluations = judge_case(case, n_judges=M_JUDGES, judge_model=JUDGE_MODEL)
        judge_rows.extend(judges_to_long_rows(row, evaluations))
        write_judge_csv(judge_path, judge_rows)
judge_df = pd.read_csv(judge_path) if judge_path and Path(judge_path).exists() else pd.DataFrame(judge_rows)
judge_path, judge_df.head()

(None,
 Empty DataFrame
 Columns: []
 Index: [])

In [7]:
eval_prefix = bench_dir / f'eval_{stamp}'
artifacts = build_scorecard(results_path, judge_path=judge_path, out_prefix=eval_prefix)
scorecard_df, category_df, long_df = build_scorecard_frames(results_path, judge_path=judge_path)
artifacts, scorecard_df.head(), category_df

({'wide': WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench/eval_20260617_004403.csv'),
  'long': WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench/eval_20260617_004403.long.csv'),
  'category': WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench/eval_20260617_004403.category.csv'),
  'markdown': WindowsPath('p:/AI_asistent/ITMO/blocksnet-agent/docs/bench/eval_20260617_004403.md')},
                           id category  \
 0  q01_block_603_development        A   
 1          q02_block_7_needs        A   
 2    q03_block_711_provision        A   
 3   q04_city_hotel_placement        B   
 4   q05_city_pitch_placement        B   
 
                                         question  n  D1_median  D1_iqr  \
 0           Что нужно разместить в квартале 603?  3        1.0     0.0   
 1                  Чего не хватает в квартале 7?  3        1.0     0.0   
 2  Какова обеспеченность квартала 711 сервисами?  3        1.0     0.0   
 3                Где разместить

In [8]:
display_cols = ['id', 'category', 'n', 'D1_median', 'D2_median', 'D3_median', 'D4_median', 'composite_median', 'calls_median', 'wasted_calls_median']
display(scorecard_df[[col for col in display_cols if col in scorecard_df.columns]])
display(category_df)

plot_df = category_df.set_index('category')[[col for col in ['D1_median', 'D2_median', 'D3_median', 'D4_median'] if col in category_df.columns]]
ax = plot_df.plot(kind='bar', ylim=(0, 1), figsize=(8, 4), title='BNA-Eval by category')
ax.set_ylabel('score')

,id,category,n,D1_median,D2_median,D3_median,D4_median,composite_median,calls_median,wasted_calls_median
0,q01_block_603_development,A,3,1.0,0.750000,0.518519,0.756000,0.756130,10.0,0.0
1,q02_block_7_needs,A,3,1.0,0.833333,0.518519,0.756000,0.765463,11.0,0.0
2,q03_block_711_provision,A,3,1.0,0.666667,0.518519,0.756000,0.735296,10.0,0.0
3,q04_city_hotel_placement,B,3,1.0,0.666667,0.666667,0.756000,0.772333,7.0,0.0
4,q05_city_pitch_placement,B,3,1.0,0.750000,0.592593,0.556000,0.602593,7.0,0.0
5,q06_city_pharmacy_gaps,B,3,1.0,0.833333,0.824074,0.710000,0.858056,5.0,0.0
6,q07_pedestrian_streets,E/D,3,1.0,0.833333,0.518519,0.510000,0.715463,10.0,0.0
7,q08_healthcare_gaps,C,3,0.5,0.833333,0.444444,0.556000,0.583444,5.0,0.0
8,q09_lowest_transport_accessibility,C,3,1.0,1.000000,0.444444,0.556667,0.758611,5.0,0.0
9,q10_least_connected_blocks,C,3,1.0,1.000000,0.444444,0.390000,0.708611,4.0,0.0


,category,n,D1_median,D1_iqr,D2_median,D2_iqr,D3_median,D3_iqr,D4_median,D4_iqr,composite_median,composite_iqr,calls_median,calls_iqr,wasted_calls_median,run_dirs
0,A,12,1.0,0.00,0.708333,0.128846,0.518519,0.037037,0.756,0.039000,0.745713,0.035917,10.0,1.5,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...
1,B,9,1.0,0.00,0.750000,0.166667,0.666667,0.226852,0.756,0.283000,0.772333,0.172324,7.0,2.0,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...
2,C,12,1.0,0.25,1.000000,0.016667,0.444444,0.000000,0.390,0.166333,0.708611,0.038611,4.0,1.0,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...
3,D,9,1.0,0.50,0.800000,0.097222,0.518519,0.148148,0.510,0.285000,0.605833,0.094676,5.0,3.0,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...
4,E,9,1.0,0.50,0.666667,0.119444,0.444444,0.037037,0.710,0.249667,0.658778,0.096250,9.0,3.5,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...
5,E/D,3,1.0,0.00,0.833333,0.000000,0.518519,0.148148,0.510,0.200000,0.715463,0.087037,10.0,1.0,0.0,p:\AI_asistent\ITMO\blocksnet-agent\outputs\ru...


Text(0, 0.5, 'score')

In [9]:
QUESTION_ID = results_df.iloc[0]['id'] if len(results_df) else None
REPEAT = int(results_df.iloc[0]['repeat']) if len(results_df) else 1

selected = results_df[(results_df['id'] == QUESTION_ID) & (results_df['repeat'] == REPEAT)]
if len(selected):
    run_dir = Path(selected.iloc[0]['run_dir'])
    log_md = run_dir / 'run_log.md'
    print(log_md)
    print(log_md.read_text(encoding='utf-8') if log_md.exists() else 'run_log.md не найден')
    if len(judge_df):
        display(judge_df[(judge_df['id'] == QUESTION_ID) & (judge_df['repeat'].astype(str) == str(REPEAT))][['criterion', 'judge', 'score', 'evidence', 'commentary']])

p:\AI_asistent\ITMO\blocksnet-agent\outputs\run_20260617-004409-2425fc\run_log.md
# Run Log

- Run ID: `20260617-004409-2425fc`
- Model: `openai/gpt-4o`
- Started: `2026-06-16T21:44:09+00:00`
- Finished: `2026-06-16T21:44:57+00:00`

## Question
Что нужно разместить в квартале 603?

## Confidence
- Scored (авторитетный): `0.62`
- Self-reported (модель): `0.9`

## Tool Calls
| Tool | Args | Observation |
|---|---|---|
| `compute_service_provision` | `{'service_type': 'key', 'accessibility_minutes': 15}` | Батч-обеспеченность набора 'key' (34 сервисов):<br>\| service \| threshold_min \| strong \| weak \| full_blocks \| partial_blocks \| missing_blocks \| status \|<br>\|---\|---:\|---:\|---:\|---:\|---:\|---:\|---\|<br>\| bar \| 20 \| 0.046 \| 0.046 \| 2 \| 4 \| 897 \| слабая \| |
| `compute_mean_accessibility` | `{}` | Средняя доступность (out=True) вычислена.<br>Мин: 28.34, макс: 254.04, среднее: 49.91, медиана: 33.82.<br>Топ-5 наиболее доступных (наименьшее время), mean_accessibility:<b

In [10]:
print('results:', results_path)
print('judge:', judge_path)
for name, path in artifacts.items():
    print(name, path)

results: p:\AI_asistent\ITMO\blocksnet-agent\docs\bench\results_20260617_004403.csv
judge: None
wide p:\AI_asistent\ITMO\blocksnet-agent\docs\bench\eval_20260617_004403.csv
long p:\AI_asistent\ITMO\blocksnet-agent\docs\bench\eval_20260617_004403.long.csv
category p:\AI_asistent\ITMO\blocksnet-agent\docs\bench\eval_20260617_004403.category.csv
markdown p:\AI_asistent\ITMO\blocksnet-agent\docs\bench\eval_20260617_004403.md


## Как читать скоркарту

`D1` — понимание вопроса, `D2` — выбор инструментов, `D3` — корректность использования инструментов, `D4` — выводы и обоснование. Основной разрез — медиана и IQR по нескольким прогонам; `composite` нужен только для быстрого сравнения версий.